In [1]:
!pip install sagemaker boto3 --upgrade -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.67.1 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
tensorflow 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.4 which is incompatible.
tensorflow 2.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.


In [3]:
import sagemaker, boto3

sess   = sagemaker.session.Session()
role   = sagemaker.get_execution_role()
region = boto3.Session().region_name
BUCKET = "crop-disease-ruchita"   # ← your bucket name

print("Role:", role)
print("Region:", region)
print("Bucket:", BUCKET)

AttributeError: module 'sagemaker' has no attribute 'session'

In [1]:
!pip uninstall -y sagemaker
!pip install sagemaker==2.224.1 --quiet

Found existing installation: sagemaker 3.8.0
Uninstalling sagemaker-3.8.0:
  Successfully uninstalled sagemaker-3.8.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.67.1 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
tensorflow 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.4 which is incompatible.
opentelemetry-proto 1.41.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.


In [1]:
import sagemaker
import boto3

sess = sagemaker.session.Session()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

print("Working ✅")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Working ✅


In [4]:
import sagemaker, boto3

sess   = sagemaker.Session()
role   = sagemaker.get_execution_role()
region = boto3.Session().region_name
BUCKET = "crop-disease-ruchita"   # ← your bucket name

print("Role:", role)
print("Region:", region)
print("Bucket:", BUCKET)%autocall

Role: arn:aws:iam::100908259520:role/service-role/AmazonSageMaker-ExecutionRole-20260420T150633
Region: us-east-1
Bucket: crop-disease-ruchita


NameError: name 'autocall' is not defined

In [12]:
%%writefile train.py

import os, json, torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

train_dir = os.environ.get("SM_CHANNEL_TRAIN",      "/opt/ml/input/data/train")
val_dir   = os.environ.get("SM_CHANNEL_VALIDATION", "/opt/ml/input/data/validation")
model_dir = os.environ.get("SM_MODEL_DIR",          "/opt/ml/model")

NUM_CLASSES = 38
EPOCHS      = 6         # ← increased from 3 to 10
BATCH       = 32
LR          = 0.001

tfm_train = transforms.Compose([
    transforms.Resize((224, 224)),        # ← increased from 128 to 224
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),        # ← added rotation augmentation
    transforms.ColorJitter(0.3,0.3,0.3), # ← added color augmentation
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
tfm_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_ds = datasets.ImageFolder(train_dir, transform=tfm_train)
val_ds   = datasets.ImageFolder(val_dir,   transform=tfm_val)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)

print(f"Classes : {len(train_ds.classes)}")
print(f"Train   : {len(train_ds)} images")
print(f"Val     : {len(val_ds)}   images")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {device}")

model = models.mobilenet_v2(weights="IMAGENET1K_V1")
model.classifier[1] = nn.Linear(model.last_channel, NUM_CLASSES)
model = model.to(device)

# freeze base layers, only train classifier first
for param in model.features.parameters():
    param.requires_grad = False

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

for epoch in range(EPOCHS):
    # unfreeze all layers after epoch 3
    if epoch == 3:
        for param in model.features.parameters():
            param.requires_grad = True
        optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_dl:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct      += (out.argmax(1) == labels).sum().item()
        total        += labels.size(0)

    train_acc = correct / total * 100
    print(f"Epoch {epoch+1}/{EPOCHS}  Loss: {running_loss/total:.4f}  "
          f"Train Acc: {train_acc:.1f}%")

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_dl:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            val_correct += (out.argmax(1) == labels).sum().item()
            val_total   += labels.size(0)
    print(f"           Val Acc: {val_correct/val_total*100:.1f}%")
    scheduler.step()

os.makedirs(model_dir, exist_ok=True)
torch.save(model.state_dict(), os.path.join(model_dir, "model.pth"))
with open(os.path.join(model_dir, "class_index.json"), "w") as f:
    json.dump(train_ds.class_to_idx, f)

print("✅ Model saved!")

Overwriting train.py


In [ ]:
from sagemaker.pytorch import PyTorch

estimator = PyTorch(
    entry_point      = "train.py",
    role             = role,
    framework_version= "2.1",
    py_version       = "py310",
    instance_count   = 1,
    instance_type    = "ml.m5.xlarge",   # CPU — no GPU quota needed
    volume_size      = 30,
    max_run          = 7200,
    output_path      = f"s3://{BUCKET}/model-output/",
    sagemaker_session= sess,
    hyperparameters  = {},
)

estimator.fit(
    inputs={
        "train":      sagemaker.inputs.TrainingInput(f"s3://{BUCKET}/dataset/train/"),
        "validation": sagemaker.inputs.TrainingInput(f"s3://{BUCKET}/dataset/val/"),
    },
    wait=True,
    logs=True,
)

print("\n✅ Training complete!")
print("Job name:", estimator.latest_training_job.name)

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: pytorch-training-2026-04-20-13-50-23-981


2026-04-20 13:50:24 Starting - Starting the training job...
2026-04-20 13:50:41 Starting - Preparing the instances for training...
2026-04-20 13:51:05 Downloading - Downloading input data.........
2026-04-20 13:52:25 Downloading - Downloading the training image...
2026-04-20 13:53:11 Training - Training image download completed. Training in progress..bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
/opt/conda/lib/python3.10/site-packages/paramiko/pkey.py:100: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
/opt/conda/lib/python3.10/site-packages/paramiko/transport.py:259: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,
2026-04

In [ ]:
%%writefile inference.py

import os, json, torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import io

NUM_CLASSES = 38
CLASS_INDEX = {}

def model_fn(model_dir):
    global CLASS_INDEX
    m = models.mobilenet_v2(weights=None)
    m.classifier[1] = nn.Linear(m.last_channel, NUM_CLASSES)
    m.load_state_dict(torch.load(
        os.path.join(model_dir, "model.pth"), map_location="cpu"))
    m.eval()
    with open(os.path.join(model_dir, "class_index.json")) as f:
        raw = json.load(f)
        CLASS_INDEX = {v: k for k, v in raw.items()}
    return m

def input_fn(body, content_type):
    img = Image.open(io.BytesIO(body)).convert("RGB")
    tfm = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    return tfm(img).unsqueeze(0)

def predict_fn(tensor, model):
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
        idx   = probs.argmax().item()
    return {"label": CLASS_INDEX.get(idx,"Unknown"),
            "confidence": round(probs[idx].item()*100, 1),
            "is_healthy": "healthy" in CLASS_INDEX.get(idx,"").lower()}

def output_fn(pred, accept):
    import json
    return json.dumps(pred), "application/json"

In [11]:
from sagemaker.pytorch import PyTorchModel

model = PyTorchModel(
    model_data        = estimator.model_data,
    role              = role,
    framework_version = "2.1",
    py_version        = "py310",
    entry_point       = "inference.py",
)

predictor = model.deploy(
    initial_instance_count = 1,
    instance_type          = "ml.m5.large",
)

print("✅ Endpoint:", predictor.endpoint_name)
print("⚠️  COPY THIS NAME for Lambda!")

INFO:sagemaker:Repacking model artifact (s3://crop-disease-ruchita/model-output/pytorch-training-2026-04-20-11-28-21-955/output/model.tar.gz), script artifact (None), and dependencies ([]) into single tar.gz file located at s3://sagemaker-us-east-1-100908259520/pytorch-inference-2026-04-20-13-02-03-848/model.tar.gz. This may take some time depending on model size...
INFO:sagemaker:Creating model with name: pytorch-inference-2026-04-20-13-02-05-825
INFO:sagemaker:Creating endpoint-config with name pytorch-inference-2026-04-20-13-02-06-558
INFO:sagemaker:Creating endpoint with name pytorch-inference-2026-04-20-13-02-06-558


------!✅ Endpoint: pytorch-inference-2026-04-20-13-02-06-558
⚠️  COPY THIS NAME for Lambda!


In [17]:
predictor.delete_endpoint()
print("✅ Endpoint deleted — no more charges")

INFO:sagemaker:Deleting endpoint configuration with name: pytorch-inference-2026-04-20-13-02-06-558
INFO:sagemaker:Deleting endpoint with name: pytorch-inference-2026-04-20-13-02-06-558


✅ Endpoint deleted — no more charges
